# Minería de Datos · Sesión 8: Clasificación II
## Naive Bayes y K-Vecinos más Cercanos (KNN) frente al Árbol de Decisión
**Proyecto:** Análisis del Hurto de Vehículos y Capacidad Policial en Colombia  
**Docente:** Esteban Ernesto Morales Castro · UNIMINUTO Sede Ibagué · 2026-2  
**Instrumento:** Taller Práctico / Torneo de Clasificación (Unidad 2 - PEA)

---

### Resumen de la Misión:
En la Sesión 7 construimos los árboles de decisión oficiales del proyecto, estableciendo una línea base de **82,0% - 82,7%** de Accuracy.  
En esta sesión ponemos a competir al Árbol de Decisión contra dos grandes familias algorítmicas sobre nuestro dataset integrado (`dataset_integrado.csv` con $54.428$ filas):
1. **Gaussian Naive Bayes (GaussianNB):** Modelo probabilístico basado en el Teorema de Bayes con supuesto de independencia condicional.
2. **K-Nearest Neighbors (KNN):** Algoritmo perezoso (lazy) basado en distancias euclidianas y votación vecinal.
3. **Escalamiento Riguroso:** Demostración empírica de por qué KNN exige `StandardScaler` dentro de `Pipeline` para evitar fuga de datos.
4. **Exploración del Hiperparámetro k:** Análisis de sobreajuste ($k=1$), punto de equilibrio y subajuste ($k=51$), contrastando esquemas de ponderación.
5. **Veredicto Técnico Oficial:** Decisión argumentada para la Policía Nacional ponderando exactitud, interpretabilidad y costo computacional.


## 1. Importación de Librerías y Módulos del Proyecto


In [1]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Carga modular de la arquitectura del proyecto
sys.path.append('../src')
from clasificacion import TorneoClasificacion

print("✅ Entorno de la Sesión 8 inicializado correctamente.")


✅ Entorno de la Sesión 8 inicializado correctamente.


## 2. Carga y Diagnóstico del Dataset Integrado


In [2]:
torneo = TorneoClasificacion('../dataset/dataset_integrado.csv')
df = torneo.cargar()
print(f"Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df[['AÑO', 'DEPARTAMENTO', 'MUNICIPIO', 'TIPO_VEHICULO', 'VEHICULOS_REGISTRADOS', 'VEHICULOS_RECUPERADOS', 'VEHICULOS_HURTADOS']].head()


Dimensiones del dataset: 54,428 filas × 8 columnas


## 3. Fundamentos Teóricos I · Teorema de Bayes con Datos de Colombia
El Teorema de Bayes permite calcular la probabilidad posterior de un evento delictivo condicionada por la presencia de señales operativas:
$$P(\text{Hurto Activo} \mid \text{Señal Operativa}) = \frac{P(\text{Señal Operativa} \mid \text{Hurto Activo}) \cdot P(\text{Hurto Activo})}{P(\text{Señal Operativa})}$$


In [3]:
res_bayes = torneo.demostrar_teorema_bayes()
for k, v in res_bayes.items():
    print(f"• {k:<38}: {v}")

print("\nInterpretación de Seguridad Pública:")
print(f"Probabilidad a priori de hurto en Colombia: {res_bayes['p_previa_hurto']*100:.1f}%")
print(f"Probabilidad posterior ante operativos de recuperación: {res_bayes['p_posterior_bayes']*100:.1f}%")
print(f"Factor de actualización de riesgo: {res_bayes['factor_actualizacion']}x veces superior.")


• total_observaciones                   : 54428
• municipios_con_hurto                  : 24117
• municipios_sin_hurto                  : 30311
• p_previa_hurto                        : 0.4431
• sensibilidad_p_senial_dado_hurto      : 0.5363
• falso_positivo_p_senial_dado_no_hurto : 0.2663
• p_posterior_bayes                     : 0.6157
• factor_actualizacion                  : 1.39

Interpretación de Seguridad Pública:
Probabilidad a priori de hurto en Colombia: 44.3%
Probabilidad posterior ante operativos de recuperación: 61.6%
Factor de actualización de riesgo: 1.39x veces superior.


## 4. Fundamentos Teóricos II · Distancia Euclidiana a Mano (Tolima)
Cálculo paso a paso de la distancia euclidiana entre un nuevo caso en **Ibagué** y municipios candidatos del **Tolima (Melgar, Espinal, Flandes)**:
$$d(x, y) = \sqrt{\sum_{i=1}^{p} (x_i - y_i)^2}$$


In [4]:
dist_demo = torneo.calcular_distancia_manual()
print("Caso Nuevo Evaluado:", dist_demo["caso_nuevo"])
print("\n--- DISTANCIAS CRUDAS (SIN ESCALAR) ---")
for cand, datos in dist_demo["distancias_crudas"].items():
    print(f"• {cand:<24}: d = {datos['distancia']} (Diferencias: {datos['diferencias']})")

print("\n--- DISTANCIAS CON STANDARDSCALER ---")
for cand, datos in dist_demo["distancias_escaladas"].items():
    print(f"• {cand:<24}: d_escalada = {datos['distancia_escalada']}")


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Caso Nuevo Evaluado: {'AÑO': 2019.0, 'VEHICULOS_REGISTRADOS': 450.0, 'VEHICULOS_RECUPERADOS': 15.0}

--- DISTANCIAS CRUDAS (SIN ESCALAR) ---
• Candidato A (Melgar)    : d = 330.18 (Diferencias: {'AÑO': 0.0, 'VEHICULOS_REGISTRADO

## 5. El Tablero de Juego Compartido (Partición de Laboratorio)
Partición estratificada $75\%$ Train ($40.821$ casos) y $25\%$ Test ($13.607$ casos) con semilla fija $42$.


In [5]:
X_train, X_test, y_train, y_test = torneo.preparar_datos(test_size=0.25, random_state=42)
print(f"Entrenamiento (Train): {X_train.shape[0]:,} registros")
print(f"Prueba (Test):         {X_test.shape[0]:,} registros")
print(f"Proporción de clase en Test: {dict(y_test.value_counts(normalize=True).round(3))}")


Entrenamiento (Train): 40,821 registros
Prueba (Test):         13,607 registros
Proporción de clase en Test: {0: np.float64(0.557), 1: np.float64(0.443)}


## 6. El Torneo de Clasificación: Los Tres Rivales
Entrenamiento y medición de tiempos con `time.perf_counter()` para Árbol, GaussianNB, KNN escalado y KNN sin escalar.


In [6]:
df_torneo = torneo.ejecutar_torneo()
print(df_torneo[['tiempo_fit_s', 'tiempo_pred_s', 'acc_train', 'acc_test', 'brecha_gen_pct', 'f1_score']])


                      tiempo_fit_s tiempo_pred_s  ... brecha_gen_pct f1_score
Arbol (prof=4)               0.014        0.0011  ...           0.38   0.8275
Naive Bayes                 0.0036        0.0006  ...           0.79   0.6971
KNN (k=5)                    0.013        0.0582  ...           3.66   0.8117
KNN (k=5 sin escalar)       0.0089        0.0344  ...           4.07   0.8034

[4 rows x 6 columns]


## 7. Exploración Sistemática del Hiperparámetro k en KNN
Evaluamos $k \in [1, 3, 5, 11, 21, 31, 51]$ para estudiar el balance entre memorización y generalización.


In [7]:
df_k = torneo.explorar_k_knn(k_valores=[1, 3, 5, 11, 21, 31, 51])
print(df_k[['ponderacion', 'k', 'acc_train', 'acc_test', 'f1_test', 'brecha_pct', 'diagnostico']])


   ponderacion   k  acc_train  ...  f1_test  brecha_pct                 diagnostico
0      uniform   1     0.8789  ...   0.7871        9.15  Sobreajuste / Memorización
1      uniform   3     0.8545  ...   0.8019        5.29          Zona de equilibrio
2      uniform   5     0.8479  ...   0.8117        3.66          Zona de equilibrio
3      uniform  11     0.8450  ...   0.8226        2.30          Zona de equilibrio
4      uniform  21     0.8431  ...   0.8247        1.89          Zona de equilibrio
5      uniform  31     0.8406  ...   0.8261        1.51          Zona de equilibrio
6      uniform  51     0.8387  ...   0.8260        1.32             Subajuste suave
7     distance   1     0.8789  ...   0.7871        9.15  Sobreajuste / Memorización
8     distance   3     0.8876  ...   0.7964        9.14          Zona de equilibrio
9     distance   5     0.8908  ...   0.8035        8.75          Zona de equilibrio
10    distance  11     0.8975  ...   0.8117        8.63          Zona de equ

## 8. Generación de Gráficas Oficiales
Generamos figuras en alta resolución guardadas en `analisis/figuras/`:
1. `fig_sesion8_torneo_comparacion.png`
2. `fig_sesion8_efecto_k_knn.png`
3. `fig_sesion8_matrices_confusion.png`


In [8]:
figs = torneo.generar_graficos('../analisis/figuras')
for f in figs:
    print("Figura oficial generada:", f)


✅ Figuras de la Sesión 8 guardadas en: ../analisis/figuras
Figura oficial generada: ../analisis/figuras/fig_sesion8_torneo_comparacion.png
Figura oficial generada: ../analisis/figuras/fig_sesion8_efecto_k_knn.png
Figura oficial generada: ../analisis/figuras/fig_sesion8_matrices_confusion.png


## 9. Veredicto Técnico Escrito y Síntesis Comparativa

| Criterio Metodológico | Árbol de Decisión (prof=4) | Gaussian Naive Bayes | K-Nearest Neighbors (k=5) |
| :--- | :---: | :---: | :---: |
| **Accuracy Test** | **82,71% (Ganador)** | 70,46% | 81,13% (Escalado) |
| **F1-Score Ponderado** | **82,75%** | 69,71% | 81,17% |
| **Tiempo de Fit** | 0,0121 s | **0,0036 s (Rápido)** | 0,0118 s |
| **Tiempo de Inferencia** | **0,0015 s (Instantáneo)** | 0,0022 s | 0,5510 s (Lento) |
| **Explicabilidad Legal** | **Alta (4 preguntas Sí/No)** | Baja (Probabilidades) | Media (Analogía) |
| **¿Requiere Escalamiento?** | No | No | **Sí, crítico (cae a 80,3%)** |
| **Veredicto Final** | **Recomendado para Producción** | Línea Base Permanente | Descartado por Costo Inferencia |


In [9]:
print(torneo.generar_veredicto_tecnico())


VEREDICTO TÉCNICO OFICIAL · RECOMENDACIÓN PARA EL SISTEMA POLICIAL (SESIÓN 8):
1. RECOMENDACIÓN PRINCIPAL: Se recomienda desplegar el ÁRBOL DE DECISIÓN (profundidad 4) como modelo operativo central.
2. FUNDAMENTO DE RENDIMIENTO: El Árbol alcanzó 82,7% de Accuracy en Test y F1-Score de 82,8%, superando holgadamente a Naive Bayes (70,5%).
3. VENTAJA DE EXPLICABILIDAD: A diferencia de KNN y Bayes, el Árbol genera 4 umbrales físicos de Sí/No que los comandantes de estación pueden auditar y justificar ante la ley.
4. EFICIENCIA OPERATIVA EN PRODUCCIÓN: El Árbol predice en tiempo O(profundidad), mientras KNN obliga a buscar vecinos entre 54.428 registros en cada consulta.
5. LECCIÓN CRÍTICA DE ESCALA: El experimento confirmó que KNN sin StandardScaler degrada su exactitud (cae a 80,3%), demostrando que 'la distancia manda a escalar'.
6. ROL DE NAIVE BAYES: Aunque GaussianNB es el más veloz entrenando (0,0038s), sus bajas métricas reflejan la violación del supuesto de independencia entre parq